[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-imbalanced.ipynb)

# Handling Imbalanced Data

*AIBits Academy · Machine Learning End To End · Practical ML · New*

Fraud, churn, disease, defaults — nearly every high-value classification problem is imbalanced. A model that "does well" by accuracy can be worthless in production.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Packages that Colab does not ship by default (a no-op if already installed)
%pip install -q imbalanced-learn

> **📋 Real-World Case Study — Credit Card Default & Company Bankruptcy Prediction**
>
> Both problems share the same structural challenge: genuine defaulters/bankruptcies are rare (often under 5% of cases) but represent almost all the real business cost. A bankruptcy-prediction model trained naively on raw historical filings will, just like the Axis Bank example, learn to predict "solvent" for everyone and still score highly on accuracy — while missing every company actually worth flagging for credit review. This is precisely the scenario where the resampling and threshold-tuning techniques on this page earn their keep over a naively-trained baseline.

## The Accuracy Trap, Revisited

You've already seen the 93%/7% Axis Bank default dataset in the Model Evaluation chapter. A model that predicts "no default" for every single applicant scores 93% accuracy while catching zero actual defaulters. This page covers the techniques that fix the *training process itself*, not just the evaluation metric.

## Three Families of Fix

| Approach | Mechanism | Trade-off |
|---|---|---|
| **Resampling** | Change the training data's class balance directly | Oversampling risks overfitting the minority class; undersampling discards majority-class information |
| **Class weighting** | Penalise minority-class errors more heavily in the loss function | No data is discarded or fabricated; requires a model that supports sample/class weights |
| **Threshold tuning** | Move the classification decision threshold away from the default 0.5 | Works on any already-trained probabilistic classifier; doesn't fix a poorly-calibrated model |

## 1. Resampling — SMOTE

Random oversampling duplicates minority-class rows verbatim — risking the model memorising exact duplicates. **SMOTE** (Synthetic Minority Oversampling Technique) instead generates *synthetic* minority samples by interpolating between a real minority point and one of its nearest minority neighbours:

$$x_{\text{new}} = x_i + \lambda\cdot(x_{\text{neighbour}}-x_i) \quad \text{where } \lambda \sim \text{Uniform}(0,1)$$

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# HDFC credit card fraud: 97% legitimate, 3% fraud
X, y = make_classification(n_samples=8000, n_features=10, n_informative=6,
                           weights=[0.97,0.03], random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# SMOTE must ONLY be applied to the training fold, never to test data
X_tr_res, y_tr_res = SMOTE(random_state=42).fit_resample(X_tr, y_tr)
print(f"Before SMOTE: {dict(zip(*[list(x) for x in __import__('numpy').unique(y_tr, return_counts=True)]))}")
print(f"After  SMOTE: {dict(zip(*[list(x) for x in __import__('numpy').unique(y_tr_res, return_counts=True)]))}")

clf = LogisticRegression(max_iter=1000).fit(X_tr_res, y_tr_res)
print(classification_report(y_te, clf.predict(X_te), digits=3))

> **⚠ SMOTE Inside CV Folds Only**
>
> Exactly like target encoding and scaling, SMOTE must be fit *only* on each training fold, never on validation/test data — synthesising minority points using neighbours that include test-set points leaks test information into training. Use `imblearn.pipeline.Pipeline` (not plain sklearn Pipeline) so SMOTE is correctly resampled fresh inside every CV fold.

## Try It — Generate a SMOTE Point Yourself

55 majority-class points (blue) vs. 6 real minority-class points (red) — roughly the 9:1 imbalance ratio from the code above. Click the button to generate one synthetic minority point at a time using the exact formula from this section: pick a real minority point xᵢ, pick one of its nearest minority neighbours, and interpolate with λ~Uniform(0,1).

## 2. Class Weighting — No Synthetic Data Needed

Instead of changing the data, reweight the loss function so misclassifying a minority sample costs more:

$$J_{\text{weighted}}(\theta) = -\frac{1}{m}\sum_i w_{y_i}\cdot\big[y_i\log(\hat{y}_i)+(1-y_i)\log(1-\hat{y}_i)\big] \quad w_{\text{minority}} = \frac{n_{\text{majority}}}{n_{\text{minority}}}\ (\text{typical default})$$

In [ ]:
# One-line fix — no resampling, no synthetic data, works with the ORIGINAL data
clf_weighted = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_tr, y_tr)
print(classification_report(y_te, clf_weighted.predict(X_te), digits=3))
# class_weight='balanced' sets w_c = n_samples / (n_classes * count(c)) automatically
# Available on: LogisticRegression, SVC, DecisionTree/RandomForest, and most sklearn classifiers

## 3. Threshold Tuning — Free, Post-Hoc

The default 0.5 decision threshold is arbitrary — it implicitly assumes false positives and false negatives cost the same, which is almost never true for fraud/default/disease detection. Moving the threshold trades precision for recall along the model's existing ROC curve, with zero retraining:

In [ ]:
from sklearn.metrics import precision_recall_curve

probs = clf.predict_proba(X_te)[:,1]
precisions, recalls, thresholds = precision_recall_curve(y_te, probs)

# Find the threshold that guarantees at least 70% recall (catch 70%+ of fraud)
import numpy as np
valid = recalls[:-1] >= 0.70
best_idx = np.argmax(precisions[:-1][valid]) if valid.any() else 0
chosen_threshold = thresholds[valid][best_idx] if valid.any() else 0.5
print(f"Threshold for 70%+ recall: {chosen_threshold:.3f} (default is always 0.5)")

## Choosing an Approach

| Situation | Recommended first move |
|---|---|
| Model supports class_weight natively | Start with class weighting — simplest, no data fabrication |
| Severe imbalance (<1% minority), model doesn't support weights | SMOTE (or SMOTE + undersampling combo like SMOTEENN) |
| Already have a trained probabilistic model, need a quick fix | Threshold tuning — zero retraining cost |
| Business has an explicit cost ratio (e.g., 1 missed fraud = ₹50k, 1 false alert = ₹200) | Cost-sensitive threshold chosen directly from that ratio, not an arbitrary recall target |

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · The accuracy trap

A model that always predicts "legitimate" (0) scores high accuracy on rare-fraud data. Compute `naive_acc` (accuracy of always predicting 0) and `naive_recall` (recall of the fraud class for that model).

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, recall_score
rng = np.random.default_rng(0)
y = (rng.random(2000) < 0.03).astype(int)
naive_pred = np.zeros_like(y)
naive_acc = naive_recall = None   # TODO


In [ ]:
try:
    check("accuracy looks great (> 0.95)", naive_acc > 0.95)
    check("recall is zero", naive_recall == 0)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.metrics import accuracy_score, recall_score
rng = np.random.default_rng(0)
y = (rng.random(2000) < 0.03).astype(int)
naive_pred = np.zeros_like(y)
naive_acc = accuracy_score(y, naive_pred)
naive_recall = recall_score(y, naive_pred)

```

</details>

### Exercise 2 · Medium · Class weights lift recall

Fit two logistic regressions on the training split — plain and `class_weight="balanced"` — and store the fraud-class recall of each on the test split in `recall_plain` and `recall_bal`.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
X, y = make_classification(n_samples=6000, n_features=8, n_informative=4, weights=[0.97, 0.03], flip_y=0.01, class_sep=0.8, random_state=1)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, random_state=1)
recall_plain = recall_bal = None   # TODO


In [ ]:
try:
    check("balanced weighting raises recall", recall_bal > recall_plain)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
X, y = make_classification(n_samples=6000, n_features=8, n_informative=4, weights=[0.97, 0.03], flip_y=0.01, class_sep=0.8, random_state=1)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, random_state=1)
recall_plain = recall_score(y_te, LogisticRegression(max_iter=1000).fit(X_tr, y_tr).predict(X_te))
recall_bal = recall_score(y_te, LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_tr, y_tr).predict(X_te))

```

</details>

### Exercise 3 · Stretch · SMOTE on the training set only

Oversample the **training** data with `SMOTE(random_state=0)` into `X_res, y_res` (never the test set!). Store the class counts before and after as `before` and `after` (dicts via `collections.Counter`).

In [ ]:
from collections import Counter
from imblearn.over_sampling import SMOTE
before = after = None   # TODO (use X_tr, y_tr from the previous exercise)


In [ ]:
try:
    check("training set was imbalanced", before[0] > 10 * before[1])
    check("SMOTE balanced it", after[0] == after[1])
    check("majority untouched", after[0] == before[0])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from collections import Counter
from imblearn.over_sampling import SMOTE
before = Counter(y_tr)
X_res, y_res = SMOTE(random_state=0).fit_resample(X_tr, y_tr)
after = Counter(y_res)

```

Resampling before the split leaks synthetic copies of test points into training and inflates every metric.

</details>

---
*Back to the course: **Machine Learning End To End → Handling Imbalanced Data**.*